# Bayes Tabanlı Restoran Menü Optimizasyonu
**Öğrenci:** Ahmet Ayvalık
**Numara:** 230212041
**Ders:** Üretken Yapay Zeka
**Açıklama:** Restoran sipariş verisi üzerinde Bayes Teoremi
ile koşullu olasılık hesaplanarak otomatik menü paketi Üretimi.

## 1. Kütüphane Kurulumu ve İçe Aktarma
Açıklama: Projede kullanılan tüm Python kütüphaneleri.
- pandas: veri manipülasyonu
- mlxtend: Apriori algoritması (araç olarak)
- matplotlib/seaborn: görselleştirme
- numpy: sayısal işlemler

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style("whitegrid")
print("Tüm kütüphaneler başarıyla yüklendi.")

Tüm kütüphaneler başarıyla yüklendi.


## 2. Veri Seti: Nereden Alındı ve İçeriği
Açıklama: Kaggle'dan indirilen gerçek restoran sipariş verisi.
Kaynak, kolon açıklamaları ve temel istatistikler burada tanıtılıyor.

In [2]:
df = pd.read_csv(
    'veri/comprehensive-restaurant-orders-popular-items-high-value-transactions-patterns.csv',
    sep='|'
)
print(f"Veri boyutu: {df.shape}")
print()
print(f"Kolonlar: {list(df.columns)}")
print()
print("İlk 5 satır:")
df.head()

Veri boyutu: (12097, 8)

Kolonlar: ['order_details_id', 'order_id', 'order_date', 'order_time', 'item_id', 'item_name', 'category', 'price']

İlk 5 satır:


,order_details_id,order_id,order_date,order_time,item_id,item_name,category,price
0,1,1,1/1/23,11:38:36 AM,109,Korean Beef Bowl,Asian,17.95
1,94,38,1/1/23,5:03:00 PM,109,Korean Beef Bowl,Asian,17.95
2,110,45,1/1/23,5:55:48 PM,109,Korean Beef Bowl,Asian,17.95
3,125,52,1/1/23,6:53:02 PM,109,Korean Beef Bowl,Asian,17.95
4,175,76,1/2/23,12:21:17 PM,109,Korean Beef Bowl,Asian,17.95


In [3]:
print("=== VERİ SETİ ÖZET BİLGİSİ ===")
print(f"Toplam kayıt:        {len(df):,}")
print(f"Benzersiz sipariş:   {df['order_id'].nunique():,}")
print(f"Benzersiz ürün:      {df['item_name'].nunique()}")
print(f"Kategoriler:         {df['category'].unique().tolist()}")
print(f"Eksik değer toplamı: {df.isnull().sum().sum()}")
print()
print("Fiyat istatistikleri:")
print(df['price'].describe().round(2))

=== VERİ SETİ ÖZET BİLGİSİ ===
Toplam kayıt:        12,097
Benzersiz sipariş:   5,343
Benzersiz ürün:      32
Kategoriler:         ['Asian', 'Italian', 'Mexican', 'American']
Eksik değer toplamı: 0

Fiyat istatistikleri:
count    12097.00
mean        13.16
std          3.99
min          5.00
25%         10.50
50%         13.95
75%         16.50
max         19.95
Name: price, dtype: float64


## 3. Veri Ön İşleme (Preprocessing)
Açıklama: Ham verideki 4 sorun ve çözümleri:
1. Pipe (|) ayraç → pd.read_csv(sep='|') ile çözüldü
2. Tarih formatı '1/1/23' → pd.to_datetime ile parse edildi
3. Saat formatı '11:38 AM' → sayısal saate dönüştürüldü
4. Ham satır → Bayes için transaction listesi oluşturuldu

In [4]:
# Tarih dönüşümü
df['order_date_parsed'] = pd.to_datetime(df['order_date'], format='%m/%d/%y')
df['month'] = df['order_date_parsed'].dt.month
df['dayofweek'] = df['order_date_parsed'].dt.dayofweek

# Saat dönüşümü
df['order_time_parsed'] = pd.to_datetime(df['order_time'], format='%I:%M:%S %p')
df['hour'] = df['order_time_parsed'].dt.hour

# Saat dilimi kategorisi
def get_time_slot(hour):
    if 12 <= hour <= 14:  return 'Öğle'
    elif 15 <= hour <= 17: return 'Öğleden Sonra'
    elif 18 <= hour <= 21: return 'Akşam'
    elif 22 <= hour <= 23: return 'Gece'
    else:                  return 'Diğer'

df['time_slot'] = df['hour'].apply(get_time_slot)

print("ön işleme tamamlandı.")
print("Türetilen yeni alanlar: month, dayofweek, hour, time_slot")
print()
print("Saat dilimi dağılımı:")
print(df['time_slot'].value_counts())

ön işleme tamamlandı.
Türetilen yeni alanlar: month, dayofweek, hour, time_slot

Saat dilimi dağılımı:
time_slot
Öğle             4173
Akşam            3846
Öğleden Sonra    3133
Diğer             629
Gece              316
Name: count, dtype: int64


## 4. Veri Artırma (Data Augmentation)
Açıklama: Ham veriyi zenginleştirerek Bayes analizine
daha fazla boyut kattık. Türetilen alanlar:
- hour: Bayes zaman filtresi için
- month: Aylık Bayes faktörü değişimi
- dayofweek: Hafta içi/sonu bazlı analiz
- time_slot: Saat dilimi segmentasyonu
- transaction_list: Bayes P(B|A) hesabı için

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Kategori dağılımı
cat_counts = df['category'].value_counts()
axes[0].pie(cat_counts.values, labels=cat_counts.index,
            autopct='%1.1f%%', colors=['#1E2761','#F96167','#F59E0B','#10B981'])
axes[0].set_title('Mutfak Kategorisi Dağılımı', fontweight='bold')

# Saat dilimi dağılımı
slot_counts = df.groupby('time_slot')['order_id'].nunique()
slot_order = ['Öğle','Öğleden Sonra','Akşam','Gece']
slot_counts = slot_counts.reindex([s for s in slot_order if s in slot_counts.index])
axes[1].bar(slot_counts.index, slot_counts.values,
            color=['#1E2761','#CADCFC','#F96167','#64748B'])
axes[1].set_title('Saat Dilimine Göre Sipariş Yoğunluğu', fontweight='bold')
axes[1].set_xlabel('Saat Dilimi')
axes[1].set_ylabel('Sipariş Sayısı')
for i, v in enumerate(slot_counts.values):
    axes[1].text(i, v + 30, str(v), ha='center', fontweight='bold')

# Aylık sipariş dağılımı
month_counts = df.groupby('month')['order_id'].nunique()
month_names = {1: 'Ocak', 2: 'Şubat', 3: 'Mart'}
axes[2].bar([month_names.get(m, m) for m in month_counts.index],
            month_counts.values, color=['#1E2761','#F96167','#10B981'])
axes[2].set_title('Aylık Sipariş Dağılımı', fontweight='bold')
axes[2].set_xlabel('Ay')
axes[2].set_ylabel('Sipariş Sayısı')
for i, v in enumerate(month_counts.values):
    axes[2].text(i, v + 10, str(v), ha='center', fontweight='bold')

plt.suptitle('Veri Artırma (Augmentation) → Türetilen Alanlarön Görselleştirilmesi',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('ciktilar/gorseller/augmentation_gorsellestirme.png', dpi=150, bbox_inches='tight')
plt.show()
print("Augmentation görselleştirmesi tamamlandı.")

Augmentation görselleştirmesi tamamlandı.


## 5. Bayes Teoremi: Projenin Teorik Temeli
Açıklama:
P(B|A) = P(A|B) → P(B) / P(A)

Bu projede:
- Prior P(B): Bir ürünön genel sipariş olasılığı (bağlamdan bağımsız)
- Kanıt A: Başka bir ürün sipariş edildi
- Posterior P(B|A): Kanıt sonrası güncellenmiş olasılık = Confidence
- Bayes Faktörü: Posterior / Prior = Lift

Apriori bu projede sadece bir araçtır.
Asıl çıktı Bayes Posterior değerleri ve Bayes Faktörleridir.

In [6]:
# Transaction listesi oluştur (Bayes girdi formatı)
transactions = df.groupby('order_id')['item_name'].apply(list).tolist()
print(f"Transaction sayısı: {len(transactions)}")
print(f"Örnek transaction: {transactions[0]}")

# TransactionEncoder
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
te_df = pd.DataFrame(te_array, columns=te.columns_)

# Apriori → sıklık analizi aracı
frequent_itemsets = apriori(te_df, min_support=0.01, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
print()
print(f"Bulunan sık öğe kümeleri: {len(frequent_itemsets)}")
print(f"İkili kümeler: {len(frequent_itemsets[frequent_itemsets['length']==2])}")

Transaction sayısı: 5343
Örnek transaction: ['Korean Beef Bowl']

Bulunan sık öğe kümeleri: 64
İkili kümeler: 32


## 6. Bayes Hesaplama: Prior → Posterior → Bayes Faktörü
Açıklama:
Her kural için hesaplanan Bayes metrikleri:
- prior_B: P(B) = consequent'in genel sipariş oranı
- posterior_B_given_A: P(B|A) = confidence değeri
- bayes_factor: posterior / prior = lift değeri
- belief_update_pct: ınancı göncelleme yüzdesi
  ((posterior - prior) / prior → 100)
- interpretation: Güçlü / Orta / Zayıf Bayes kanıtı

In [7]:
# Association rules → ham kurallar
rules_raw = association_rules(
    frequent_itemsets, metric='lift', min_threshold=1.05
)
rules_raw = rules_raw[
    rules_raw['antecedents'].apply(len) == 1
][rules_raw['consequents'].apply(len) == 1].copy()

# Bidirectional deduplication
seen = set()
dedup_idx = []
for idx, row in rules_raw.iterrows():
    pair = frozenset([list(row['antecedents'])[0],
                      list(row['consequents'])[0]])
    if pair not in seen:
        seen.add(pair)
        dedup_idx.append(idx)
rules_raw = rules_raw.loc[dedup_idx].copy()

# Bayes metrikleri hesapla
def calculate_bayes_metrics(rules_df, freq_items):
    results = []
    single_items = freq_items[freq_items['length'] == 1].copy()
    single_items['item'] = single_items['itemsets'].apply(
        lambda x: list(x)[0]
    )
    support_map = dict(zip(single_items['item'], single_items['support']))

    for _, rule in rules_df.iterrows():
        ant = list(rule['antecedents'])[0]
        con = list(rule['consequents'])[0]
        prior_B = support_map.get(con, rule['consequent support'])
        posterior_B_given_A = rule['confidence']
        bayes_factor = posterior_B_given_A / prior_B if prior_B > 0 else 0
        belief_update = ((posterior_B_given_A - prior_B) / prior_B) * 100

        if bayes_factor > 2.5:
            interpretation = "Güçlü Bayes Kanıtı"
        elif bayes_factor > 1.5:
            interpretation = "Orta Bayes Kanıtı"
        else:
            interpretation = "Zayıf Bayes Kanıtı"

        results.append({
            'antecedent': ant,
            'consequent': con,
            'prior_B': round(prior_B, 4),
            'posterior_B_given_A': round(posterior_B_given_A, 4),
            'bayes_factor': round(bayes_factor, 3),
            'belief_update_pct': round(belief_update, 1),
            'support': round(rule['support'], 4),
            'interpretation': interpretation
        })
    return pd.DataFrame(results)

bayes_df = calculate_bayes_metrics(rules_raw, frequent_itemsets)
bayes_df = bayes_df.sort_values('bayes_factor', ascending=False).reset_index(drop=True)
print(f"Toplam Bayes kuralı: {len(bayes_df)}")
print()
print("İlk 5 kural:")
bayes_df.head()
bayes_df.to_csv('ciktilar/tablolar/bayes_kurallari.csv', index=False)

Toplam Bayes kuralı: 23

İlk 5 kural:


## 7. Başarı Metrikleri
Açıklama:
- Toplam Bayes kuralı sayısı
- Maksimum Bayes Faktörü (Prior'dan ne kadar uzaklaştık)
- Maksimum Posterior P(B|A) değeri
- Maksimum ınancı Güncellemesi (+%)
Güçlü/Orta/Zayıf kanıt dağılımı

In [8]:
print("=" * 50)
print("BAYES MODELİ BAŞARI METRİKLERİ")
print("=" * 50)
print(f"Toplam Bayes Kuralı:          {len(bayes_df)}")
print(f"Maksimum Bayes Faktörü:       {bayes_df['bayes_factor'].max():.2f}x")
print(f"Maksimum Posterior P(B|A):    %{bayes_df['posterior_B_given_A'].max()*100:.1f}")
print(f"Maksimum Prior P(B):          %{bayes_df['prior_B'].max()*100:.1f}")
print(f"Maksimum ınancı Güncellemesi:  +%{bayes_df['belief_update_pct'].max():.0f}")
print()
print("Kanıt Sınıfı Dağılımı:")
print(bayes_df['interpretation'].value_counts().to_string())

# Metrik tablosu → tüm kurallar
print()
print("Tüm Bayes Kuralları (ilk 10):")
display_cols = ['antecedent','consequent','prior_B',
                'posterior_B_given_A','bayes_factor',
                'belief_update_pct','interpretation']
bayes_df[display_cols].head(10)

BAYES MODELİ BAŞARI METRİKLERİ
Toplam Bayes Kuralı:          23
Maksimum Bayes Faktörü:       1.47x
Maksimum Posterior P(B|A):    %13.8
Maksimum Prior P(B):          %11.2
Maksimum ınancı Güncellemesi:  +%47

Kanıt Sınıfı Dağılımı:
interpretation
Zayıf Bayes Kanıtı    23

Tüm Bayes Kuralları (ilk 10):


,antecedent,consequent,prior_B,posterior_B_given_A,bayes_factor,belief_update_pct,interpretation
0,Chips & Salsa,Steak Torta,0.0882,0.1295,1.469,46.9,Zayıf Bayes Kanıtı
1,Tofu Pad Thai,Chicken Burrito,0.0825,0.1089,1.319,31.9,Zayıf Bayes Kanıtı
2,Cheeseburger,Steak Torta,0.0882,0.1163,1.319,31.9,Zayıf Bayes Kanıtı
3,Chips & Salsa,Edamame,0.1115,0.1384,1.241,24.1,Zayıf Bayes Kanıtı
4,Mac & Cheese,French Fries,0.1024,0.1267,1.237,23.7,Zayıf Bayes Kanıtı
5,Korean Beef Bowl,Spaghetti & Meatballs,0.0861,0.1060,1.231,23.1,Zayıf Bayes Kanıtı
6,Chicken Burrito,Cheeseburger,0.1046,0.1270,1.214,21.4,Zayıf Bayes Kanıtı
7,Tofu Pad Thai,Steak Torta,0.0882,0.1070,1.214,21.4,Zayıf Bayes Kanıtı
8,Edamame,Cheeseburger,0.1046,0.1258,1.203,20.3,Zayıf Bayes Kanıtı
9,Orange Chicken,French Fries,0.1024,0.1228,1.200,20.0,Zayıf Bayes Kanıtı


## 8. Grafik Analizleri
Açıklama:
Grafik 1 → Bayes Posterior P(B|A) Dağılımı (Bar Chart)
  En yüksek bar = en güçlü Bayes göncellemesi.
  Tüm çubuklar Prior'dan büyük → gerçek bağlamsal ilişki var.

Grafik 2 → Bayes Faktörü Azalımlı Dağılımı (Line Chart)
  1.0 çizgisi sönır: altönda Bayes göncellemesi yok.
  Eğri hızla düşüyor → az kural çok büyük güce sahip (Pareto).

Grafik 3 → Prior vs Posterior Karşılaştırması
  Bayes göncellemesinin ne kadar güçlü çalıştığını gösteriyor.

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
# Grafik 1: Top 5 Bayes Posterior
top5 = bayes_df.head(5)
labels = [f"{r['antecedent']}\n→ {r['consequent']}"
          for _, r in top5.iterrows()]
bars = axes[0].barh(labels,
                    top5['posterior_B_given_A'] * 100,
                    color='#1E2761', edgecolor='white')
for bar, val in zip(bars, top5['posterior_B_given_A'] * 100):
    axes[0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'%{val:.1f}', va='center', fontweight='bold', color='#1E2761')
axes[0].set_xlabel('Bayes Posterior P(B|A) (%)')
axes[0].set_title('Top 5 Kural → Bayes Posterior Değerleri\n(Confidence = P(B|A))',
                  fontweight='bold')
axes[0].axvline(x=bayes_df['prior_B'].mean() * 100,
                color='#F96167', linestyle='--', linewidth=2,
                label=f"Ortalama Prior: %{bayes_df['prior_B'].mean()*100:.1f}")
axes[0].legend()
axes[0].set_xlim(0, 100)
# Grafik 2: Bayes Faktörü Dağılımı
plot_df = bayes_df.head(20).reset_index(drop=True)
axes[1].plot(range(1, len(plot_df)+1),
             plot_df['bayes_factor'],
             color='#F96167', linewidth=3, marker='o',
             markersize=8, markerfacecolor='white',
             markeredgecolor='#F96167', markeredgewidth=2)
axes[1].axhline(y=1.0, color='#64748B', linestyle='--',
                linewidth=2, label='Bayes sınırı (Faktör=1.0)')
axes[1].axhline(y=1.2, color='#F59E0B', linestyle=':',
                linewidth=2, label='Minimum eşik (1.2x)')
axes[1].fill_between(range(1, len(plot_df)+1),
                     plot_df['bayes_factor'], 1.0,
                     alpha=0.15, color='#F96167')
axes[1].set_xlabel('Kural Sırası (Azalımlı)')
axes[1].set_ylabel('Bayes Faktörü (Posterior/Prior)')
axes[1].set_title('Bayes Faktörü Azalımlı Dağılımı\n(İlk 20 Kural)',
                  fontweight='bold')
axes[1].legend()
plt.suptitle('Bayes Modeli → Grafik Analizleri',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('ciktilar/gorseller/bayes_grafik_analizi.png', dpi=150, bbox_inches='tight')
plt.show()


In [10]:
top8 = bayes_df.head(8)
x = np.arange(len(top8))
width = 0.35
fig, ax = plt.subplots(figsize=(14, 7))
bars1 = ax.bar(x - width/2, top8['prior_B'] * 100, width,
               label='Prior P(B) → Önceki inancı',
               color='#CADCFC', edgecolor='#1E2761', linewidth=1.5)
bars2 = ax.bar(x + width/2, top8['posterior_B_given_A'] * 100, width,
               label='Posterior P(B|A) → Sonraki inancı',
               color='#1E2761', edgecolor='#1E2761', linewidth=1.5)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'%{bar.get_height():.1f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold',
            color='#1E2761')
labels8 = [f"{r['antecedent'][:15]}\n→ {r['consequent'][:15]}"
           for _, r in top8.iterrows()]
ax.set_xticks(x)
ax.set_xticklabels(labels8, fontsize=9)
ax.set_ylabel('Olasılık (%)')
ax.set_title('Bayes Güncellemesi: Prior → Posterior Karşılaştırması\n'
             'Mavi çubuklar ne kadar büyükse Bayes güncellemesi o kadar güçlü',
             fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig('ciktilar/gorseller/prior_posterior_karsilastirma.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Otomatik Menü Paketi Üretimi (Üretken Yan)
Açıklama:
Bayes Faktörü gücüne göre 3 karar senaryosu:
- Faktör > 2.5 → Paket sun, indirim yapma
  (Müşteri zaten birlikte alıyor)
- Faktör 1.5–2.5 → öndirimli paket oluştur (%8 indirim)
- Faktör 1.2–1.5 → Küçük indirimle dene
- Faktör < 1.2 → Pakete ekleme

In [11]:
PRICE_MAP = {
    'Shrimp Scampi': 19.95, 'Chicken Parmesan': 17.95,
    'Korean Beef Bowl': 17.95, 'Pork Ramen': 17.95,
    'Meat Lasagna': 17.95, 'Spaghetti & Meatballs': 17.95,
    'Eggplant Parmesan': 16.95, 'Orange Chicken': 16.50,
    'Cheese Lasagna': 15.50, 'Mushroom Ravioli': 15.50,
    'Salmon Roll': 14.95, 'Steak Burrito': 14.95,
    'Fettuccine Alfredo': 14.50, 'Spaghetti': 14.50,
    'Tofu Pad Thai': 14.50, 'Cheeseburger': 13.95,
    'Steak Tacos': 13.95, 'Steak Torta': 13.95,
    'Chicken Burrito': 12.95, 'Hamburger': 12.95,
    'Chicken Torta': 11.95, 'California Roll': 11.95,
    'Chicken Tacos': 11.95, 'Cheese Quesadillas': 10.50,
    'Veggie Burger': 10.50, 'Chips & Guacamole': 9.00,
    'Hot Dog': 9.00, 'Potstickers': 9.00,
    'Chips & Salsa': 7.00, 'French Fries': 7.00,
    'Mac & Cheese': 7.00, 'Edamame': 5.00,
}

CAT_MAP = df.groupby('item_name')['category'].first().to_dict()

def generate_package_name(row):
    if row['bayes_factor'] > 2.5:
        return f"Şef'in {row['antecedent']} Kombinasyonu"
    elif row['bayes_factor'] > 1.5:
        return f"{row['antecedent']} & {row['consequent']} özel Menüsı"
    else:
        return f"{row['consequent']} ile {row['antecedent']} Menüsı"

def generate_price_strategy(row, total_price):
    pct = round(row['posterior_B_given_A'] * 100)
    bf = round(row['bayes_factor'], 1)
    if row['bayes_factor'] > 2.5:
        return (f"öndirim yapma → müşteri zaten birlikte alıyor. "
                f"Normal fiyat: ${total_price:.2f}")
    elif row['bayes_factor'] > 1.5:
        disc = total_price * 0.92
        return (f"öndirimli paket sun. "
                f"Normal: ${total_price:.2f} → Paket: ${disc:.2f} (%%8 indirim)")
    else:
        disc = total_price * 0.95
        return (f"Küçük indirimle dene. "
                f"Normal: ${total_price:.2f} → Paket: ${disc:.2f} (%%5 indirim)")

packages = []
for _, row in bayes_df[bayes_df['bayes_factor'] >= 1.2].head(10).iterrows():
    ant_price = PRICE_MAP.get(row['antecedent'], 10.0)
    con_price = PRICE_MAP.get(row['consequent'], 10.0)
    total = ant_price + con_price

    # Günlük ek gelir tahmini
    total_orders = df['order_id'].nunique()
    total_days = df['order_date_parsed'].nunique()
    ant_daily = (df[df['item_name'] == row['antecedent']]
                 ['order_id'].nunique() / total_days)
    remaining = 1 - row['posterior_B_given_A']
    daily_potential = ant_daily * remaining * con_price

    packages.append({
        'paket_ismi': generate_package_name(row),
        'antecedent': row['antecedent'],
        'consequent': row['consequent'],
        'bayes_faktoru': row['bayes_factor'],
        'posterior': row['posterior_B_given_A'],
        'belief_update': row['belief_update_pct'],
        'paket_toplam_fiyat': round(total, 2),
        'fiyat_stratejisi': generate_price_strategy(row, total),
        'gunluk_ek_gelir': round(daily_potential, 2),
        'yillik_potansiyel': round(daily_potential * 365, 0),
    })

packages_df = pd.DataFrame(packages)
print("OTOMATIK ÜRETILEN MENÜ PAKETLERİ")
print("=" * 60)
for i, pkg in packages_df.iterrows():
    print()
    print('=' * 60)
    print(f"PAKET {i+1}: {pkg['paket_ismi']}")
    print(f"  {pkg['antecedent']} → {pkg['consequent']}")
    print(f"  Bayes Faktörü:    {pkg['bayes_faktoru']}x")
    print(f"  Posterior P(B|A): %{pkg['posterior']*100:.1f}")
    print(f"  ınancı Gönc.:      +%{pkg['belief_update']:.0f}")
    print(f"  Paket Fiyatı:     ${pkg['paket_toplam_fiyat']}")
    print(f"  Strateji:         {pkg['fiyat_stratejisi']}")
    print(f"  Günlük Ek Gelir:  ${pkg['gunluk_ek_gelir']}/gün")
    print(f"  Yıllık Potansiyel: ${pkg['yillik_potansiyel']:,.0f}/yıl")

packages_df.to_csv('ciktilar/tablolar/menu_paketleri.csv', index=False)

OTOMATIK ÜRETILEN MENÜ PAKETLERİ

PAKET 1: Steak Torta ile Chips & Salsa Menüsı
  Chips & Salsa → Steak Torta
  Bayes Faktörü:    1.469x
  Posterior P(B|A): %13.0
  ınancı Gönc.:      +%47
  Paket Fiyatı:     $20.95
  Strateji:         Küçük indirimle dene. Normal: $20.95 → Paket: $19.90 (%%5 indirim)
  Günlük Ek Gelir:  $60.45/gün
  Yıllık Potansiyel: $22,063/yıl

PAKET 2: Chicken Burrito ile Tofu Pad Thai Menüsı
  Tofu Pad Thai → Chicken Burrito
  Bayes Faktörü:    1.319x
  Posterior P(B|A): %10.9
  ınancı Gönc.:      +%32
  Paket Fiyatı:     $27.45
  Strateji:         Küçük indirimle dene. Normal: $27.45 → Paket: $26.08 (%%5 indirim)
  Günlük Ek Gelir:  $69.49/gün
  Yıllık Potansiyel: $25,366/yıl

PAKET 3: Steak Torta ile Cheeseburger Menüsı
  Cheeseburger → Steak Torta
  Bayes Faktörü:    1.319x
  Posterior P(B|A): %11.6
  ınancı Gönc.:      +%32
  Paket Fiyatı:     $27.9
  Strateji:         Küçük indirimle dene. Normal: $27.90 → Paket: $26.50 (%%5 indirim)
  Günlük Ek Gelir:  $76.

## 10. Aylık Trend Analizi
Açıklama:
Her ay için ayrı Bayes analizi yapılıyor.
Bayes Faktörü artıyorsa → kombinasyon güçleniyor (menüde kalsön).
Bayes Faktörü düşüyorsa → kombinasyon zayıflıyor (paketten çıkar).
Bu klasik sistemlerin yapamadığı dinamik Bayes izlemesidir.

In [12]:
monthly_results = {}
for month in [1, 2, 3]:
    month_df = df[df['month'] == month]
    if len(month_df) < 100:
        continue
    m_trans = month_df.groupby('order_id')['item_name'].apply(list).tolist()
    te_m = TransactionEncoder()
    te_arr_m = te_m.fit_transform(m_trans)
    te_df_m = pd.DataFrame(te_arr_m, columns=te_m.columns_)
    try:
        freq_m = apriori(te_df_m, min_support=0.01, use_colnames=True)
        freq_m['length'] = freq_m['itemsets'].apply(len)
        rules_m = association_rules(freq_m, metric='lift', min_threshold=1.0)
        rules_m = rules_m[
            rules_m['antecedents'].apply(len) == 1
        ][rules_m['consequents'].apply(len) == 1].copy()
        rules_m['antecedent'] = rules_m['antecedents'].apply(lambda x: list(x)[0])
        rules_m['consequent'] = rules_m['consequents'].apply(lambda x: list(x)[0])
        monthly_results[month] = rules_m
    except:
        pass
# Top 3 kombinasyon için trend hesapla
top3_combos = [
    ('Steak Torta', 'Chips & Salsa'),
    ('Hamburger', 'French Fries'),
    ('Korean Beef Bowl', 'Edamame'),
]
month_names = {1: 'Ocak', 2: 'Şubat', 3: 'Mart'}
fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#F96167', '#1E2761', '#10B981']
for (ant, con), color in zip(top3_combos, colors):
    factors = []
    months_found = []
    for month, rules_m in monthly_results.items():
        match = rules_m[
            (rules_m['antecedent'] == ant) &
            (rules_m['consequent'] == con)
        ]
        if len(match) > 0:
            prior = match['consequent support'].values[0]
            posterior = match['confidence'].values[0]
            bf = posterior / prior if prior > 0 else 0
            factors.append(round(bf, 2))
            months_found.append(month_names[month])
    if factors:
        ax.plot(months_found, factors, color=color, linewidth=3,
                marker='o', markersize=10, markerfacecolor='white',
                markeredgecolor=color, markeredgewidth=3,
                label=f"{ant} → {con}")
        for m, f in zip(months_found, factors):
            ax.annotate(f'{f:.2f}x', (m, f),
                       textcoords='offset points', xytext=(0, 12),
                       ha='center', fontweight='bold', color=color, fontsize=10)
ax.axhline(y=1.2, color='#64748B', linestyle='--',
           linewidth=1.5, alpha=0.7, label='Minimum Bayes eşiği (1.2x)')
ax.set_ylabel('Bayes Faktörü (Posterior/Prior)', fontsize=12)
ax.set_title('Aylık Bayes Faktörü Trendi (Ocak–Mart 2023)\n'
             'Artan eğri = güçlenen Bayes kanıtı', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(0.8, 3.5)
plt.tight_layout()
plt.savefig('ciktilar/gorseller/aylik_trend.png', dpi=150, bbox_inches='tight')
plt.show()


## 11. Baseline Karşılaştırması
Açıklama:
Klasik yöntem: En popüler ürünü herkese öner → P(B) sabit kalır.
Bayes yöntemi: Sipariş bağlamına göre P(B|A) hesapla.

Fark: Klasik sistem Chips & Salsa'yı %8.6 oranında önerirken,
Bayes sistemi Steak Torta bağlamında %79 oranında öneriyor.
Bu 9 katlık bir artış → Bayes'in gücü budur.

In [13]:
# Baseline: en popüler 5 ürün
top_items = df.groupby('item_name')['order_id'].nunique()
top_items = top_items.sort_values(ascending=False).head(5)

print("BASELINE SİSTEM → En Popüler 5 ürün (Herkese Aynı öneri):")
for item, count in top_items.items():
    pct = count / df['order_id'].nunique() * 100
    print(f"  {item:<30} {count:>5} sipariş (%{pct:.1f})")

print()
print("="*60)
print("BAYES SİSTEMİ → Bağlamsal öneri Karşılaştırması:")
print("="*60)
print()
print(f"{'Kombinasyon':<40} {'Prior':>8} {'Posterior':>10} {'Bayes':>8} {'Göncelleme':>12}")
print("-"*80)
for _, row in bayes_df.head(5).iterrows():
    print(f"{row['antecedent'][:20]} → {row['consequent'][:15]:<20}"
          f" %{row['prior_B']*100:>5.1f}"
          f"    %{row['posterior_B_given_A']*100:>5.1f}"
          f"    {row['bayes_factor']:>5.2f}x"
          f"    +%{row['belief_update_pct']:>6.0f}")

print()
print(f"En güçlü Bayes göncellemesi: +%{bayes_df['belief_update_pct'].max():.0f}")
print(f"Bu, klasik sistemin {bayes_df['bayes_factor'].max():.1f}x üzerinde performans gösterdiği anlamına gelir.")

BASELINE SİSTEM → En Popüler 5 ürün (Herkese Aynı öneri):
  Edamame                          596 sipariş (%11.2)
  Hamburger                        595 sipariş (%11.1)
  Korean Beef Bowl                 566 sipariş (%10.6)
  Cheeseburger                     559 sipariş (%10.5)
  French Fries                     547 sipariş (%10.2)

BAYES SİSTEMİ → Bağlamsal öneri Karşılaştırması:

Kombinasyon                                 Prior  Posterior    Bayes   Göncelleme
--------------------------------------------------------------------------------
Chips & Salsa → Steak Torta          %  8.8    % 13.0     1.47x    +%    47
Tofu Pad Thai → Chicken Burrito      %  8.2    % 10.9     1.32x    +%    32
Cheeseburger → Steak Torta          %  8.8    % 11.6     1.32x    +%    32
Chips & Salsa → Edamame              % 11.2    % 13.8     1.24x    +%    24
Mac & Cheese → French Fries         % 10.2    % 12.7     1.24x    +%    24

En güçlü Bayes göncellemesi: +%47
Bu, klasik sistemin 1.5x üzerinde perfo

## 12. Tahmini Gelecek Kazanç Analizi
Açıklama:
Formül: Günlük sipariş → (1 → Posterior) → ürün fiyatı → 365
Kalan müşteri = Bayes sisteminin henüz kazanamadığı oran.
Bu müşterileri pakete yönlendirmek ek gelir sağlar.

In [14]:
print("TAHMİNİ KAZANÇ ANALİZİ")
print("=" * 60)
print(f"{'Paket':<40} {'Günlük':>10} {'Yıllık':>12}")
print("-" * 62)
total_annual = 0
for _, pkg in packages_df.head(5).iterrows():
    print(f"{pkg['paket_ismi'][:38]:<40}"
          f" ${pkg['gunluk_ek_gelir']:>7.2f}"
          f"  ${pkg['yillik_potansiyel']:>9,.0f}")
    total_annual += pkg['yillik_potansiyel']

print("-" * 62)
print(f"{'TOPLAM TAHMİNİ YILLIK EK GELİR':40}"
      f"            ${total_annual:>9,.0f}")
print()
print("NOT: Hesaplama yalnızca mevcut sipariş verisine dayanmaktadır.")
print("     Gerçek kazanç müşteri davranışına göre değişebilir.")

TAHMİNİ KAZANÇ ANALİZİ
Paket                                        Günlük       Yıllık
--------------------------------------------------------------
Steak Torta ile Chips & Salsa Menüsı     $  60.45  $   22,063
Chicken Burrito ile Tofu Pad Thai Menü   $  69.49  $   25,366
Steak Torta ile Cheeseburger Menüsı      $  76.57  $   27,947
Edamame ile Chips & Salsa Menüsı         $  21.44  $    7,827
French Fries ile Mac & Cheese Menüsı     $  30.57  $   11,156
--------------------------------------------------------------
TOPLAM TAHMİNİ YILLIK EK GELİR                      $   94,359

NOT: Hesaplama yalnızca mevcut sipariş verisine dayanmaktadır.
     Gerçek kazanç müşteri davranışına göre değişebilir.


## 13. Sonuç
Açıklama:
Bu projede Bayes Teoremi kullanılarak:
1. Prior P(B) → Posterior P(B|A) göncellemesi gerçekleştirildi
2. Bayes Faktörü (Lift = Posterior/Prior) hesaplandı
3. Güçlü Bayes kanıtöna sahip kombinasyonlardan otomatik menü paketi üretildi
4. Aylık trend ile dinamik Bayes izlemesi yapıldı
5. Tahmini yıllık ek gelir potansiyeli hesaplandı

Klasik sistemden farkı:
- Prior sabit kalmaz → sipariş bağlamıyla göncellenir
- Üretken çıktı: isim + fiyat stratejisi + kazanç tahmini

In [15]:
print("=" * 60)
print("PROJE SONUÇ ÖZETİ")
print("=" * 60)
print(f"Öğrenci   : Ahmet Ayvalık → 230212041")
print(f"Ders      : Üretken Yapay Zeka")
print(f"Proje     : Bayes Tabanlı Restoran Menü Optimizasyonu")
print()
print("BAYES METRİKLERİ:")
print(f"  İşlenen Sipariş:          {df['order_id'].nunique():,}")
print(f"  Toplam Bayes Kuralı:      {len(bayes_df)}")
print(f"  Maks Bayes Faktörü:       {bayes_df['bayes_factor'].max():.2f}x")
print(f"  Maks Posterior P(B|A):    %{bayes_df['posterior_B_given_A'].max()*100:.1f}")
print(f"  Maks ınancı Güncellemesi:  +%{bayes_df['belief_update_pct'].max():.0f}")
print(f"  Üretilen Menü Paketi:     {len(packages_df)}")
print(f"  Yıllık Kazanç Potansiyeli: ${packages_df['yillik_potansiyel'].sum():,.0f}+")
print()
print("KAYNAKLAR (APA):")
print("  Agrawal, R., & Srikant, R. (1994). Fast algorithms for")
print("  mining association rules. VLDB Conference, 487-499.")
print()
print("  Han, J., Pei, J., & Kamber, M. (2011). Data mining:")
print("  Concepts and techniques (3rd ed.). Morgan Kaufmann.")

PROJE SONUÇ ÖZETİ
Öğrenci   : Ahmet Ayvalık → 230212041
Ders      : Üretken Yapay Zeka
Proje     : Bayes Tabanlı Restoran Menü Optimizasyonu

BAYES METRİKLERİ:
  İşlenen Sipariş:          5,343
  Toplam Bayes Kuralı:      23
  Maks Bayes Faktörü:       1.47x
  Maks Posterior P(B|A):    %13.8
  Maks ınancı Güncellemesi:  +%47
  Üretilen Menü Paketi:     10
  Yıllık Kazanç Potansiyeli: $221,192+

KAYNAKLAR (APA):
  Agrawal, R., & Srikant, R. (1994). Fast algorithms for
  mining association rules. VLDB Conference, 487-499.

  Han, J., Pei, J., & Kamber, M. (2011). Data mining:
  Concepts and techniques (3rd ed.). Morgan Kaufmann.
